# Pandas Comprehensive 
## Data Manipulation and Analysis

## Text Data, MultiIndex, Merging

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set display options
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 1000)

## 1. Working with Text Data

### 1.1 Dataset: Chicago Employee Records

In [ ]:
# Load the dataset
chicago = pd.read_csv("chicago.csv").dropna(how="all")
chicago.head()

In [ ]:
# Basic information
chicago.info()

In [ ]:
# Number of unique values in each column
chicago.nunique()

In [ ]:
# Convert Department to categorical
chicago["Department"] = chicago["Department"].astype("category")
chicago.info()

### 1.2 Common String Methods

In [ ]:
# String methods on Position Title
chicago["Position Title"].str.lower().head()
chicago["Position Title"].str.upper().head()
chicago["Position Title"].str.title().head()
chicago["Position Title"].str.len().head()
chicago["Position Title"].str.strip().head()

In [ ]:
# Replace text in Department column
chicago["Department"].str.replace("MGMNT", "MANAGEMENT").str.title().head()

### 1.3 Filtering with String Methods

In [ ]:
# Filtering with contains
water_workers = chicago["Position Title"].str.lower().str.contains("water")
chicago[water_workers].head()

In [ ]:
# Filtering with startswith
starts_with_civil = chicago["Position Title"].str.lower().str.startswith("civil")
chicago.loc[starts_with_civil].head()

In [ ]:
# Filtering with endswith
ends_with_iv = chicago["Position Title"].str.lower().str.endswith("iv")
chicago[ends_with_iv].head()

### 1.4 String Methods on Index and Columns

In [ ]:
# Setting Name as index
chicago_idx = pd.read_csv("chicago.csv", index_col="Name").dropna(how="all").sort_index()
chicago_idx["Department"] = chicago_idx["Department"].astype("category")
chicago_idx.head()

In [ ]:
# Clean the index
chicago_idx.index = chicago_idx.index.str.strip().str.title()
chicago_idx.head()

In [ ]:
# Clean column names
chicago_idx.columns = chicago_idx.columns.str.upper()
chicago_idx.head()

### 1.5 The split Method

In [ ]:
# Split Position Title and extract first word
chicago["Position Title"].str.split(" ").str.get(0).value_counts().head(10)

In [ ]:
# Split Name into Last and First
chicago[["Last Name", "First Name"]] = chicago["Name"].str.split(",", expand=True)
chicago.head()

In [ ]:
# Split with n parameter
chicago[["Primary Title", "Secondary Title"]] = chicago["Position Title"].str.split(" ", expand=True, n=1)
chicago.head()

In [ ]:
# Find most common first names
chicago["Name"].str.title().str.split(", ").str.get(1).str.strip().str.split(" ").str.get(0).value_counts().head(10)

## 2. MultiIndex (Hierarchical Indexing)

### 2.1 Creating MultiIndex DataFrames

In [ ]:
# Sample data for MultiIndex
sales_data = {
    'Product': ['A', 'A', 'A', 'B', 'B', 'B', 'C', 'C', 'C'],
    'Region': ['North', 'South', 'East', 'North', 'South', 'East', 'North', 'South', 'East'],
    'Sales': [100, 150, 120, 200, 180, 220, 80, 90, 85],
    'Profit': [20, 30, 25, 40, 35, 45, 15, 18, 17]
}
sales_df = pd.DataFrame(sales_data)
sales_df

In [ ]:
# Setting MultiIndex
multi_df = sales_df.set_index(['Product', 'Region'])
multi_df

In [ ]:
# MultiIndex from arrays
arrays = [
    ['A', 'A', 'B', 'B', 'C', 'C'],
    ['North', 'South', 'North', 'South', 'North', 'South']
]
index = pd.MultiIndex.from_arrays(arrays, names=['Product', 'Region'])
df_multi = pd.DataFrame({'Sales': [100, 150, 200, 180, 80, 90]}, index=index)
df_multi

### 2.2 Accessing Data in MultiIndex

In [ ]:
# Using .loc with MultiIndex
multi_df.loc['A']  # All entries for Product A

In [ ]:
# Accessing specific combination
multi_df.loc[('A', 'North')]

In [ ]:
# Using xs (cross-section)
multi_df.xs('North', level='Region')  # All entries for North region

In [ ]:
# Sorting MultiIndex
unsorted_multi = sales_df.set_index(['Region', 'Product']).sort_index(level=0)
unsorted_multi

### 2.3 Stack and Unstack

In [ ]:
multi_df

In [ ]:
# Unstack: Convert row level to column
unstacked = multi_df.unstack(level='Region')
unstacked

In [ ]:
# Stack: Convert columns to rows
stacked = unstacked.stack()
stacked

## 3. Merging DataFrames

In [ ]:
# Create sample dataframes for merging
employees = pd.DataFrame({
    'EmployeeID': [101, 102, 103, 104, 105],
    'Name': ['John', 'Jane', 'Bob', 'Alice', 'Charlie'],
    'DepartmentID': [1, 2, 1, 3, 2]
})

departments = pd.DataFrame({
    'DepartmentID': [1, 2, 3, 4],
    'DeptName': ['HR', 'IT', 'Finance', 'Marketing'],
    'Manager': ['Sarah', 'Mike', 'Lisa', 'Tom']
})

salaries = pd.DataFrame({
    'EmployeeID': [101, 102, 103, 104, 105],
    'Salary': [60000, 75000, 55000, 80000, 70000],
    'Bonus': [5000, 8000, 4000, 10000, 7000]
})

print("Employees:")
print(employees)
print("\nDepartments:")
print(departments)
print("\nSalaries:")
print(salaries)

### 3.1 Inner Join

In [ ]:
# Inner join - only matching records
inner_join = pd.merge(employees, departments, on='DepartmentID', how='inner')
inner_join

### 3.2 Left and Right Joins

In [ ]:
# Left join - all employees, matching departments
left_join = pd.merge(employees, departments, on='DepartmentID', how='left')
left_join

In [ ]:
# Right join - all departments, matching employees
right_join = pd.merge(employees, departments, on='DepartmentID', how='right')
right_join

### 3.3 Outer Join

In [ ]:
# Outer join - all records from both
outer_join = pd.merge(employees, departments, on='DepartmentID', how='outer')
outer_join

### 3.4 Merging with Different Column Names

In [ ]:
# Example with different column names
emp_alt = pd.DataFrame({
    'EmpID': [101, 102, 103, 104, 105],
    'Name': ['John', 'Jane', 'Bob', 'Alice', 'Charlie'],
    'DeptID': [1, 2, 1, 3, 2]
})

pd.merge(emp_alt, departments, left_on='DeptID', right_on='DepartmentID')

### 3.5 Merging Multiple DataFrames

In [ ]:
# Chain multiple merges
merged = employees.merge(departments, on='DepartmentID').merge(salaries, on='EmployeeID')
merged

### 3.6 Concatenation

In [ ]:
# Concatenate DataFrames (vertical stacking)
df1 = pd.DataFrame({'A': [1, 2], 'B': [3, 4]})
df2 = pd.DataFrame({'A': [5, 6], 'B': [7, 8]})
pd.concat([df1, df2], ignore_index=True)

In [ ]:
# Concatenate with keys
pd.concat([df1, df2], keys=['First', 'Second'])

## GroupBy, Input/Output, Simple Plots

## 4. GroupBy Operations

### 4.1 Basic GroupBy

In [ ]:
# Group by Department and aggregate salary
dept_stats = chicago.groupby('Department')['Employee Annual Salary'].agg(['count', 'mean', 'min', 'max'])
dept_stats.head(10)

In [ ]:
# Clean up salary column by removing $ and converting to float
chicago['Salary'] = chicago['Employee Annual Salary'].str.replace('$', '').str.replace(',', '').astype(float)
chicago.head()

In [ ]:
# Group by Department and Position Title
dept_pos_stats = chicago.groupby(['Department', 'Position Title'])['Salary'].agg(['mean', 'count'])
dept_pos_stats.head(10)

### 4.2 Aggregation Functions

In [ ]:
# Multiple aggregation functions
dept_agg = chicago.groupby('Department')['Salary'].agg([
    'count',
    'mean',
    'median',
    'std',
    ('min_salary', 'min'),
    ('max_salary', 'max'),
    ('range', lambda x: x.max() - x.min())
])
dept_agg.head(10)

In [ ]:
# Using agg with dictionary
dept_agg_dict = chicago.groupby('Department').agg({
    'Salary': ['mean', 'std', 'count'],
    'Position Title': 'count'
})
dept_agg_dict.head(10)

### 4.3 Transform and Apply

In [ ]:
# Transform - add group-level mean to each row
chicago['Dept_Mean_Salary'] = chicago.groupby('Department')['Salary'].transform('mean')
chicago[['Department', 'Salary', 'Dept_Mean_Salary']].head(10)

In [ ]:
# Calculate percentage of departmental mean
chicago['Salary_Pct_Of_Dept_Mean'] = (chicago['Salary'] / chicago['Dept_Mean_Salary']) * 100
chicago[['Department', 'Salary', 'Dept_Mean_Salary', 'Salary_Pct_Of_Dept_Mean']].head(10)

In [ ]:
# Apply custom function to each group
def top_employees(group, n=5):
    return group.nlargest(n, 'Salary')

top_by_dept = chicago.groupby('Department').apply(top_employees, n=3)
top_by_dept[['Department', 'Position Title', 'Salary']].head(10)

### 4.4 Filtering Groups

In [ ]:
# Filter groups with more than 1000 employees
large_depts = chicago.groupby('Department').filter(lambda x: len(x) > 1000)
large_depts['Department'].value_counts()

In [ ]:
# Filter groups where mean salary > 100000
high_pay_depts = chicago.groupby('Department').filter(lambda x: x['Salary'].mean() > 100000)
high_pay_depts['Department'].value_counts()

### 4.5 Pivot Tables

Pivot tables are like cross-tabulations that aggregate data based on two or more categorical variables.

In [ ]:
# Create a pivot table
pivot = chicago.pivot_table(
    values='Salary',
    index='Department',
    columns='Position Title',
    aggfunc='mean',
    fill_value=0
)
pivot.head(10)

In [ ]:
# Multiple aggregation functions in pivot
pivot_multi = chicago.pivot_table(
    values='Salary',
    index='Department',
    columns='Position Title',
    aggfunc=['mean', 'count'],
    fill_value=0
)
pivot_multi.head(10)

In [ ]:
# Pivot with margins (totals)
pivot_margins = chicago.pivot_table(
    values='Salary',
    index='Department',
    columns='Position Title',
    aggfunc='mean',
    margins=True,
    fill_value=0
)
pivot_margins.head(10)

## 5. Input/Output Operations

### 5.1 Reading CSV Files

In [ ]:
# Basic CSV read
df = pd.read_csv('chicago.csv')
df.head()

In [ ]:
# Reading CSV with specific parameters
df_custom = pd.read_csv(
    'chicago.csv',
    usecols=['Name', 'Department', 'Employee Annual Salary'],
    nrows=100,
    dtype={'Department': 'category'}
)
df_custom.info()

In [ ]:
# Handling missing values during read
df_missing = pd.read_csv('chicago.csv', na_values=['N/A', 'Unknown', ''])
df_missing.isnull().sum()

### 5.2 Writing CSV Files

In [ ]:
# Basic write
chicago_clean = chicago.drop(['Last Name', 'First Name', 'Primary Title', 'Secondary Title', 
                               'Dept_Mean_Salary', 'Salary_Pct_Of_Dept_Mean'], axis=1)
chicago_clean.to_csv('chicago_clean.csv', index=False)
print("File saved as 'chicago_clean.csv'")

In [ ]:
# Writing with custom separators
chicago_clean.to_csv('chicago_clean_pipe.csv', sep='|', index=False)
print("File saved as 'chicago_clean_pipe.csv' with pipe separator")

In [ ]:
# Write to Excel
chicago_clean.head(100).to_excel('chicago_sample.xlsx', sheet_name='Employees', index=False)
print("File saved as 'chicago_sample.xlsx'")

### 5.3 Reading and Writing Other Formats

In [ ]:
# Read JSON
import json
json_data = chicago_clean.head(5).to_json(orient='records')
df_json = pd.read_json(json_data, orient='records')
df_json

In [ ]:
# Write to Parquet (requires pyarrow)
try:
    chicago_clean.head(1000).to_parquet('chicago_sample.parquet')
    df_parquet = pd.read_parquet('chicago_sample.parquet')
    print("Parquet file read successfully")
    print(df_parquet.info())
except ImportError:
    print("pyarrow not installed, skipping parquet example")

In [ ]:
# Read from clipboard
# print("Copy some CSV data to clipboard, then uncomment:")
# df_clipboard = pd.read_clipboard()
# print(df_clipboard.head())

## 6. Simple Plots with Pandas

In [ ]:
# Set up plotting
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

### 6.1 Bar Plots

In [ ]:
# Top 10 departments by employee count
dept_counts = chicago['Department'].value_counts().head(10)
dept_counts.plot(kind='bar', figsize=(12, 6), title='Top 10 Departments by Employee Count')
plt.xlabel('Department')
plt.ylabel('Number of Employees')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 departments by average salary
dept_avg_salary = chicago.groupby('Department')['Salary'].mean().sort_values(ascending=False).head(10)
dept_avg_salary.plot(kind='bar', figsize=(12, 6), color='green', title='Top 10 Departments by Average Salary')
plt.xlabel('Department')
plt.ylabel('Average Salary ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 6.2 Histograms

In [ ]:
# Salary distribution
chicago['Salary'].plot(kind='hist', bins=50, figsize=(10, 6), title='Salary Distribution')
plt.xlabel('Salary ($)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Salary distribution by department (subset)
top_depts = chicago['Department'].value_counts().head(5).index
chicago[chicago['Department'].isin(top_depts)]['Salary'].plot(
    kind='hist', bins=30, figsize=(12, 6), alpha=0.7, title='Salary Distribution by Top Departments',
    legend=True
)
plt.xlabel('Salary ($)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

### 6.3 Box Plots

In [ ]:
# Box plot of salaries by department (top 10)
top_depts_by_count = chicago['Department'].value_counts().head(10).index
chicago[chicago['Department'].isin(top_depts_by_count)].boxplot(
    column='Salary', by='Department', figsize=(14, 8), rot=45
)
plt.title('Salary Distribution by Department (Top 10)')
plt.suptitle('')
plt.ylabel('Salary ($)')
plt.tight_layout()
plt.show()

### 6.4 Scatter Plots

In [ ]:
# Scatter plot: Salary vs Position Level (simulated)
# Create a simple position level indicator
chicago['Pos_Level'] = chicago['Position Title'].str.extract(r'(\d+)$').fillna(0).astype(int)

chicago.plot(
    kind='scatter', x='Pos_Level', y='Salary',
    figsize=(10, 6), alpha=0.3, title='Salary vs Position Level'
)
plt.xlabel('Position Level')
plt.ylabel('Salary ($)')
plt.tight_layout()
plt.show()

### 6.5 Line Plots

In [ ]:
# Average salary trend by position (simulated ordering)
pos_avg = chicago.groupby('Position Title')['Salary'].mean().sort_values(ascending=True).head(20)
pos_avg.plot(kind='line', figsize=(12, 6), marker='o', title='Average Salary for Top 20 Positions')
plt.xlabel('Position')
plt.ylabel('Average Salary ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 6.6 Pie Charts

In [ ]:
# Top 8 departments by employee count (pie chart)
top_8_depts = chicago['Department'].value_counts().head(8)
top_8_depts.plot(
    kind='pie', figsize=(10, 10), autopct='%1.1f%%',
    title='Top 8 Departments by Employee Count', legend=True
)
plt.tight_layout()
plt.show()

### 6.7 Advanced Visualizations with Subplots

In [ ]:
# Multiple plots in one figure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Department counts
dept_counts.head(10).plot(kind='bar', ax=axes[0, 0], title='Top 10 Departments - Counts')
axes[0, 0].set_ylabel('Count')

# Plot 2: Salary histogram
chicago['Salary'].plot(kind='hist', ax=axes[0, 1], bins=30, title='Salary Distribution')
axes[0, 1].set_xlabel('Salary ($)')

# Plot 3: Average salary by top departments
dept_avg_salary = chicago.groupby('Department')['Salary'].mean().sort_values(ascending=False).head(10)
dept_avg_salary.plot(kind='bar', ax=axes[1, 0], color='orange', title='Top 10 Departments - Avg Salary')
axes[1, 0].set_ylabel('Avg Salary ($)')

# Plot 4: Box plot for top 5 departments
top5_depts = dept_counts.head(5).index
chicago[chicago['Department'].isin(top5_depts)].boxplot(
    column='Salary', by='Department', ax=axes[1, 1]
)
axes[1, 1].set_title('Salary Distribution by Top 5 Departments')
axes[1, 1].set_ylabel('Salary ($)')
axes[1, 1].set_xlabel('Department')

plt.suptitle('Chicago Employee Data Analysis', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. Practical Exercises

### Exercise 1: Text Processing
Q: Create a new column `Salary_Band` that categorizes employees into 'Low', 'Medium', 'High' based on salary percentiles.

In [ ]:
# Solution
percentiles = chicago['Salary'].quantile([0.33, 0.67])

def salary_band(salary):
    if salary <= percentiles[0.33]:
        return 'Low'
    elif salary <= percentiles[0.67]:
        return 'Medium'
    else:
        return 'High'

chicago['Salary_Band'] = chicago['Salary'].apply(salary_band)
chicago['Salary_Band'].value_counts()

### Exercise 2: MultiIndex
Q: Create a MultiIndex DataFrame with Department and Position Title, showing the count and average salary.

In [ ]:
# Solution
multi_summary = chicago.groupby(['Department', 'Position Title']).agg(
    Count=('Salary', 'count'),
    Avg_Salary=('Salary', 'mean')
)
multi_summary.head(10)

### Exercise 3: Merging
Q: Create department summaries and merge with employee data to show % of total department employees.

In [ ]:
# Solution
dept_summary = chicago.groupby('Department').agg(
    Total_Employees=('Salary', 'count'),
    Avg_Salary=('Salary', 'mean')
).reset_index()

# Calculate percentage of total
total_employees = dept_summary['Total_Employees'].sum()
dept_summary['Pct_Of_Total'] = (dept_summary['Total_Employees'] / total_employees * 100).round(2)

dept_summary.head(10)

### Exercise 4: Visualization
Q: Create a dashboard with 4 plots showing different aspects of the data.

In [ ]:
# Solution
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Department distribution
dept_counts.head(10).plot(kind='barh', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Top 10 Departments by Employee Count')
axes[0, 0].set_xlabel('Number of Employees')

# 2. Salary distribution by Salary Band
chicago.groupby('Salary_Band')['Salary'].mean().plot(
    kind='bar', ax=axes[0, 1], color=['green', 'yellow', 'red']
)
axes[0, 1].set_title('Average Salary by Salary Band')
axes[0, 1].set_ylabel('Average Salary ($)')

# 3. Salary histogram
chicago['Salary'].plot(kind='hist', bins=40, ax=axes[1, 0], color='purple', alpha=0.7)
axes[1, 0].set_title('Salary Distribution')
axes[1, 0].set_xlabel('Salary ($)')
axes[1, 0].set_ylabel('Frequency')

# 4. Box plot by Department (top 5)
top5 = chicago['Department'].value_counts().head(5).index
chicago[chicago['Department'].isin(top5)].boxplot(
    column='Salary', by='Department', ax=axes[1, 1]
)
axes[1, 1].set_title('Salary Distribution by Department (Top 5)')
axes[1, 1].set_ylabel('Salary ($)')

plt.suptitle('Chicago Employee Data Dashboard', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 8. Summary

### Key Takeaways:

1. **Text Data**: Use `.str` accessor for string operations like `contains`, `split`, `replace`
2. **MultiIndex**: Hierarchical indexing for complex data structures using `set_index` and `xs`
3. **Merging**: Combine DataFrames using `merge` with different join types (`inner`, `left`, `right`, `outer`)
4. **GroupBy**: Powerful aggregation using `groupby` with `agg`, `transform`, and `apply`
5. **Input/Output**: Read/write various formats including CSV, Excel, JSON, and Parquet
6. **Visualizations**: Quick plotting with pandas `.plot()` for exploratory data analysis